In [1]:
!nvidia-smi

Fri Apr 10 10:13:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_id,device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

# Generate Rollouts

In [4]:
import torch
def genrate_rollouts(model,tokenizer,prompts,num_roll=4,max_new=256,temp=0.8):
  rollouts = []
  for prompt in prompts:
    toks = tokenizer.encode(prompt,return_tensors="pt").to(model.device)
    prompt_len = toks.shape[-1]
    with torch.no_grad():
      comp_tokens = model.generate(
          toks,
          temperature=temp,
          max_new_tokens=max_new,
          num_return_sequences=num_roll
      )
    comp_tokens = comp_tokens[:,prompt_len:]
    rollouts.append(comp_tokens)
  return rollouts

In [4]:
rollouts = genrate_rollouts(
    model,
    tokenizer,
    ["hello my name","i like to"],
    num_roll = 2,
    max_new = 16,
    temp = 0.7
)
rollouts

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[tensor([[  374,   557,   352,  5183,    13,   600,  1079,   264,  3162, 23576,
            323,   358,  1079,  8014,   304,   279],
         [  374,   502,   483,   323,   600,  1079,   220,    18,    15,  1635,
           2310,    13,   600,  1079,   458, 17847]], device='cuda:0'),
 tensor([[  990,   264,  3151,  1614,   369,   847,  2390,    11,   714,   279,
           1614,   374,   537,  2500,   304,   847],
         [  387,   264,   821, 18237,    11,   714,   358,  1513,   944,   614,
           1753,  3139,   448,   432,    11,   323]], device='cuda:0')]

In [5]:
[tokenizer.batch_decode(r.tolist()) for r in rollouts]

[[' is shayden. i am a software engineer and I am interested in the',
  ' is jill and i am 30 years old. i am an assistant'],
 [' use a specific model for my project, but the model is not available in my',
  " be a data analyst, but I don't have much experience with it, and"]]

# load dataset

In [5]:
from datasets import load_dataset
dataset_id = "HuggingFaceH4/MATH-500"
dataset = load_dataset(dataset_id,split="test")
dataset

Dataset({
    features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
    num_rows: 500
})

# Reward Functions

In [6]:
from math_answer_extractor import normalize_text, split_into_parts, answer_verifier, extract_final_answer
def grad_function(prediction, gtruth):
    results = [False]
    if prediction and gtruth:
        prediction = normalize_text(prediction)
        gtruth = normalize_text(gtruth)
        prediction = split_into_parts(prediction)
        gtruth = split_into_parts(gtruth)
        if prediction and gtruth and (len(gtruth) == len(prediction)):
            results = [answer_verifier(x, y) for x, y in zip(prediction, gtruth)]
    if all(results):
        return True
    return False

In [53]:
# test
answer = r"\( \frac{1}{2} + \frac{1}{6} \)"
ground_truth = "4/6"
grad_function(answer,ground_truth)

True

In [7]:
def correctness_reward(completions:list[str],ground_truth:list[str],**kargs)->list[float]:
  rewards = []
  for completion,answer in zip(completions,ground_truth):
    completion = extract_final_answer(completion)
    grad = grad_function(completion,answer)
    rewards.append(float(grad))
  return rewards

In [54]:
# test
completions = [answer]
ground_truth = [ground_truth]
correctness_reward(completions,ground_truth)

[0.0]

In [8]:
def format_reward(completions,**kargs):
  import re
  pattern = r"<think>.+?</think>\s*"#<answer>.+?</answer>"
  return [0.5 if re.search(pattern, c, re.DOTALL) else 0.0
            for c in completions]

In [56]:
# test
completions = [
    """
    <think>the reasoning steps</think>\n my answer is 1
    """,
    "casablanca"
    ]
format_reward(completions)

[0.5, 0.0]

# Prepare Dataset

In [9]:
def format_prompt(prompt,tokenizer):
  system_prompt = (
        "You are a helpful math assistant.\n" #personality
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n" # format
        "Then write the final result on a newline as:\n" # rules ...
        "\\boxed{ANSWER}\n\n"
    )
  conversation = [
      {"role":"system", "content":system_prompt},
      {"role":"user", "content":prompt},
  ]
  return tokenizer.apply_chat_template(conversation,add_generation_prompt=True,tokenize=False)

In [58]:
# test
print(format_prompt("what is 1+1?",tokenizer))

<|im_start|>system
You are a helpful math assistant.
When solving the problem, first write your reasoning inside <think> and </think> tags.
Then write the final result on a newline as:
\boxed{ANSWER}

<|im_end|>
<|im_start|>user
what is 1+1?<|im_end|>
<|im_start|>assistant



In [10]:
# format dataset
dataset = dataset.map(lambda x: {"formated_prompt":format_prompt(x,tokenizer)},input_columns=["problem"])
dataset[0]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n\n[asy]\nunitsize(0.8 cm);\n\ndraw((-0.5,0)--(3.5,0));\ndraw((0,-0.5)--(0,3.5));\ndraw(arc((0,0),3,0,90),red,Arrow(6));\n\ndot((0,3), red);\nlabel("$(0,3)$", (0,3), W);\ndot((3,0), red);\n[/asy]\n\nTherefore, the polar coordinates are $\\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}.$',
 'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'subject': 'Precalculus',
 'level': 2,
 'unique_id': 'test/precalculus/807.json',
 'formated_prompt': '<|im_start|>system\nYou are a helpful math assistant.\nWhen solving the problem, first write your reasoning inside <think> and </think> tags.\nThen write the final result on a newline as:

In [11]:
# split dataset
train_test_dataset = dataset.train_test_split(0.1)
train_test_dataset

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'formated_prompt'],
        num_rows: 450
    })
    test: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'formated_prompt'],
        num_rows: 50
    })
})

In [61]:
# test model on sample
prompt = train_test_dataset["test"][0]["formated_prompt"]
answer = train_test_dataset["test"][0]["answer"]

In [62]:
prompt

'<|im_start|>system\nYou are a helpful math assistant.\nWhen solving the problem, first write your reasoning inside <think> and </think> tags.\nThen write the final result on a newline as:\n\\boxed{ANSWER}\n\n<|im_end|>\n<|im_start|>user\nFind the curve defined by the equation\n\\[r^2 \\cos 2 \\theta = 4.\\](A) Line\n(B) Circle\n(C) Parabola\n(D) Ellipse\n(E) Hyperbola\n\nEnter the letter of the correct option.<|im_end|>\n<|im_start|>assistant\n'

In [63]:
ids = tokenizer.encode(prompt,return_tensors="pt").to(model.device)
response_ids = model.generate(ids,max_new_tokens=2048, temperature=0.8)
response = tokenizer.decode(response_ids.tolist())
response

["<|im_start|>system\nYou are a helpful math assistant.\nWhen solving the problem, first write your reasoning inside <think> and </think> tags.\nThen write the final result on a newline as:\n\\boxed{ANSWER}\n\n<|im_end|>\n<|im_start|>user\nFind the curve defined by the equation\n\\[r^2 \\cos 2 \\theta = 4.\\](A) Line\n(B) Circle\n(C) Parabola\n(D) Ellipse\n(E) Hyperbola\n\nEnter the letter of the correct option.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I need to find the curve defined by the equation $ r^2 \\cos 2\\theta = 4 $. Let me think about how to approach this. \n\nFirst, I remember that polar equations can sometimes be converted into Cartesian coordinates to make them easier to visualize. The equation given is in terms of $ r $ and $ \\theta $, so maybe I should convert it to Cartesian coordinates.\n\nStarting with the polar equation $ r^2 \\cos 2\\theta = 4 $. I know that in polar coordinates, $ r = \\sqrt{x^2 + y^2} $, and $ \\cos 2\\theta $ can be written using tr

In [120]:
format_reward(response)

[0.5]

In [121]:
correctness_reward(response,[answer])

[1.0]

# Validation Metrics

In [1]:
import torch
def calculate_eval_acc(model,tokenizer,eval_dataset,max_new_tokens=2048):
  model.eval()
  correct = 0
  total = 0
  for ed in eval_dataset:
    right_answer = ed["answer"]
    problem = ed["formated_prompt"]
    tokens = tokenizer.encode(problem,return_tensors="pt").to(model.device)
    with torch.no_grad():
      response = model.generate(tokens,max_new_tokens=max_new_tokens)
    prediction = tokenizer.decode(response.tolist())
    correct += correctness_reward(prediction,[right_answer])[0]
    total += 1
  return correct/total

In [137]:
# test eval function
test = dataset.select(range(2))
accuracy = calculate_eval_acc(model,tokenizer,test)
accuracy

Dataset({
    features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'formated_prompt'],
    num_rows: 2
})
Dataset({
    features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'formated_prompt'],
    num_rows: 2
})


0.5

In [2]:
# integrate it with trl as callback
from transformers import TrainerCallback,TrainingArguments, TrainerState, TrainerControl
class EvalCallback(TrainerCallback):
  def __init__(self,tokenizer,eval_dataset):
    self.tokenizer = tokenizer
    self.eval_dataset = eval_dataset
  def on_step_end(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
     if state.global_step % args.eval_steps == 0 and state.global_step > 0:
      model = kwargs["model"]
      acc = calculate_eval_acc(model,self.tokenizer,self.eval_dataset,args.max_completion_length)
      kwargs["trainer"].log({
          "eval_acc":acc,
          "step":state.global_step
      })

# Trainer

In [12]:
train_test_dataset = train_test_dataset.select_columns(["formated_prompt","answer",]).rename_columns({"formated_prompt":"prompt"})
train_test_dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 450
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 50
    })
})

In [13]:
from trl import GRPOConfig, GRPOTrainer
grpo_config = GRPOConfig(
    output_dir = "qwen3_grpo_0.6b",
    num_generations = 4,
    max_completion_length = 2048,
    temperature = 0.8,
    beta = 0, # no KL
    epsilon = 10,
    learning_rate=1e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    steps_per_generation=2,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.35,
    vllm_importance_sampling_correction=True,
    bf16=True,
    report_to="trackio"
)

/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/chat_completion/protocol.py:346: SyntaxWarning: invalid escape sequence '\e'
  "(e.g. 'abcdabcdabcd...' or '\emoji \emoji \emoji ...'). This feature "
/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/completion/protocol.py:176: SyntaxWarning: invalid escape sequence '\e'
  "(e.g. 'abcdabcdabcd...' or '\emoji \emoji \emoji ...'). This feature "


In [14]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward],
    args=grpo_config,
    callbacks=[EvalCallback(tokenizer, train_test_dataset["test"])],
    train_dataset=train_test_dataset["train"],
)

UnsupportedOperation: fileno

In [156]:
trainer.train()

TypeError: correctness_reward() got an unexpected keyword argument 'prompts'